# 01_Ingesta (Raw Layer)

## Importancion de recursos
En esta celda se cargan las librerías base (Pandas, NumPy) y el módulo centralizado utils.py. La centralización de rutas e importaciones garantiza que el pipeline sea reproducible y que no existan discrepancias en las rutas de acceso a los datos entre los distintos colaboradores del proyecto.

In [6]:
import utils
import pandas as pd
import numpy as np
import os

## Carga del dataset
Se procede a la lectura del archivo CSV original utilizando la ruta absoluta gestionada por utils.py. Cargar los datos desde una ubicación centralizada permite que el proyecto sea portable, facilitando su ejecución en diferentes entornos sin necesidad de modificar el código interno del notebook.

In [7]:
csv_path = os.path.join(utils.DATASETS_PATH, "global_movies_dataset_1950_2026.csv")
df = pd.read_csv(csv_path)

## Ordenar por año

In [ ]:
df = df.sort_values('release_year').reset_index(drop=True)

## Dividir en 5 subconjuntos
El dataset se fragmenta en 5 partes utilizando numpy.array_split. Esta operación simula un entorno real de ingeniería de datos donde la información no proviene de un único archivo estático, sino que se recibe de forma distribuida desde múltiples fuentes o dispositivos.

In [8]:
fragmentos = np.array_split(df, 5)

## Capa Raw
Este bloque establece la conexión con la base de datos de la capa Raw y persiste los fragmentos generados. Es fundamental re-instanciar cada fragmento como un DataFrame de Pandas para preservar los encabezados originales de las columnas, asegurando que el esquema de datos se mantenga íntegro desde el primer paso del pipeline.

In [16]:
conn = utils.get_db_connection("raw")
for i, frag in enumerate(fragmentos, 1):
    df_fragmento = pd.DataFrame(frag, columns=df.columns)
    tabla = f"raw_fuente_{i}"

    df_fragmento.to_sql(tabla, conn, if_exists="replace", index=False)

    anio_min = df_fragmento['release_year'].min()
    anio_max = df_fragmento['release_year'].max()
    print(f"✅ Tabla {f"raw_fuente_{i}"} creada. Registros: {len(frag)}. Periodo: {anio_min}-{anio_max}")

    print(f"✅ Fuente {i} guardada con nombres: {df.columns.tolist()[:3]}...")

conn.close()

✅ Tabla raw_fuente_1 creada. Registros: 20000. Periodo: 1950-2026
✅ Fuente 1 guardada con nombres: ['movie_id', 'title', 'release_year']...
✅ Tabla raw_fuente_2 creada. Registros: 20000. Periodo: 1950-2026
✅ Fuente 2 guardada con nombres: ['movie_id', 'title', 'release_year']...
✅ Tabla raw_fuente_3 creada. Registros: 20000. Periodo: 1950-2026
✅ Fuente 3 guardada con nombres: ['movie_id', 'title', 'release_year']...
✅ Tabla raw_fuente_4 creada. Registros: 20000. Periodo: 1950-2026
✅ Fuente 4 guardada con nombres: ['movie_id', 'title', 'release_year']...
✅ Tabla raw_fuente_5 creada. Registros: 20000. Periodo: 1950-2026
✅ Fuente 5 guardada con nombres: ['movie_id', 'title', 'release_year']...


## Creación del Archivo
Se genera el archivo extraction_log.json para dejar un registro de auditoría. Documentar el número de tablas creadas y la marca de tiempo es una práctica esencial de gobernanza que permite validar que la ingesta se completó con éxito antes de pasar a la siguiente etapa de procesamiento.

In [17]:
log_extracccion = {
    "etapa": "Extracción Raw",
    "tablas_creadas": 5,
    "fecha": utils.get_timestamp()
}

utils.save_log(utils.LOG_PATH, log_extracccion)
print(f"✅ Archivo guardado correctamente en: {utils.LOG_PATH}")

✅ Archivo guardado correctamente en: G:\Mi unidad\Drive.alumno.udg.mx\Proyectos\Proyecto_Integrador\logs\extraction_log.json
